# Detect entity networks in the company-grievances example with `itk-networks`

This notebook uses **`packages/itk-networks`** (`import itk_networks`), not the legacy
`intelligence_toolkit.detect_entity_networks` module or the Streamlit workflow. It reads
one existing repository input directly, without copying it or generating new data:

- `example_outputs/detect_entity_networks/company_grievances/company_grievances_input.csv`

That file has 12,000 rows describing companies: a `name`, five attributes that can link
companies to each other (`address`, `city`, `email`, `phone`, `owner`), two broad
grouping columns (`sector`, `country`) and five `*_grievances` columns holding counts of
complaints, the "flags" of this workflow. The same company appears on several rows, so
the model has about 3,600 entities.

The notebook walks the pipeline the way the workflow does: build the data model, embed
entity names so that similar spellings link up, identify networks of closely related
entities, change the settings without repeating the expensive steps, explore one network,
and read the flag exposure of one entity. A deterministic character-trigram embedder makes
every step offline and reproducible; the final AI report step is shown but only runs when
an OpenAI configuration is present in the environment.

## 1. Install and select a Python kernel

Use **Python 3.11 or 3.12** (the packages require `>=3.11,<3.13`). The `leiden` extra
pulls in graspologic, which splits networks larger than `max_network_entities`; this
dataset has such components, so install it. From the repository root, run once in a
terminal:

```powershell
uv venv --python 3.12 .venv-networks-demo
uv pip install --python .venv-networks-demo\Scripts\python.exe -e .\packages\itk-core -e ".\packages\itk-networks[leiden]" ipykernel
.\.venv-networks-demo\Scripts\python.exe -m ipykernel install --user --name itk-networks-demo --display-name "Python (itk-networks demo)"
```

On macOS/Linux use `.venv-networks-demo/bin/python` and slash-separated paths. Install
**both local packages together**: `itk-core` on PyPI is a different project. If you already
use the repository's own environment (`uv sync --extra dev`), its kernel works too, since
the workspace installs `itk-networks` and graspologic in development mode.

Open this notebook in VS Code or Jupyter, select the kernel, and run all cells. Jupyter's
top-level `await` runs the async `index` and `report` steps directly. Input lookup walks up
from the working directory, so the notebook runs from the repository root or from its own
folder. Displayed tables are previews; the complete frames stay in the `result` variables.

In [ ]:
# Copyright (c) 2026 Microsoft Corporation. All rights reserved.
# Licensed under the MIT license. See LICENSE file in the project.

import os
import re
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import Markdown
from IPython.display import display as show
from itk_networks import (
    AttributeTable,
    FlagFormat,
    FlagTable,
    GroupTable,
    NetworkConfig,
    NetworkDetector,
)
from itk_networks.explore import simplify, subgraph_frames

pl.Config.set_tbl_rows(25)
pl.Config.set_tbl_cols(14)
pl.Config.set_fmt_str_lengths(50)

cwd = Path.cwd().resolve()
relative_input = (
    Path("example_outputs")
    / "detect_entity_networks"
    / "company_grievances"
    / "company_grievances_input.csv"
)
repo_root = next(
    (parent for parent in (cwd, *cwd.parents) if (parent / relative_input).is_file()),
    None,
)
if repo_root is None:
    message = "Run from within the repository containing the company-grievances input."
    raise FileNotFoundError(message)
input_path = repo_root / relative_input
print("Input:", input_path.relative_to(repo_root))


def installed_version(name: str) -> str:
    try:
        return version(name)
    except PackageNotFoundError:  # dev-mode checkout without distribution metadata
        return "dev"


packages = [
    "itk-networks",
    "itk-core",
    "networkx",
    "numpy",
    "polars",
    "pydantic",
    "graspologic",
]
show(
    pl.DataFrame({
        "package": packages,
        "version": [installed_version(name) for name in packages],
    })
)

## 2. Prepare the data model

The model is built from three kinds of table, each pointing at the same frame with a
different reading of its columns:

- **`AttributeTable`**: `name` is the entity, and `address`, `city`, `email`, `phone` and
  `owner` become attribute nodes linked to it. Two companies that share a value are two
  hops apart in the graph. `sector` and `country` are deliberately left out: they are too
  broad and would pull unrelated companies into one network.
- **`FlagTable`** in `Count` format: each `*_grievances` column is a flag type and its value
  the number of complaints on that row; rows of the same company are summed.
- **`GroupTable`**: `sector` and `country` are carried through to the output frame so
  networks can be analysed by group without being detected through it.

`NetworkDetector.prepare` cleans the text (punctuation dropped, whitespace collapsed),
drops null cells, builds the attribute graph and the flags frame, and returns a
`NetworkResult` that every later stage extends. Node names use the legacy `type==value`
encoding, with entities under the reserved type `ENTITY`.

In [ ]:
frame = pl.read_csv(input_path)
show(Markdown(f"**Input: first 10 of {frame.height} rows**"))
show(frame.head(10))

attribute_columns = ["address", "city", "email", "phone", "owner"]
flag_columns = [
    "safety_grievances",
    "pay_grievances",
    "conditions_grievances",
    "treatment_grievances",
    "workload_grievances",
]
group_columns = ["sector", "country"]

attributes = AttributeTable(
    frame=frame, entity_column="name", columns=attribute_columns
)
flags = FlagTable(
    frame=frame, entity_column="name", columns=flag_columns, format=FlagFormat.Count
)
groups = GroupTable(frame=frame, entity_column="name", columns=group_columns)

prepared = NetworkDetector.prepare([attributes], [flags], [groups])
print(prepared.summary)
print("Attribute types:", prepared.node_types)
print("Indexable node types:", prepared.entity_types)
print(
    f"Most flags on one entity: {prepared.max_entity_flags}; "
    f"mean over flagged entities: {prepared.mean_flagged_flags}"
)
show(Markdown("**Attribute links: first 8 rows**"))
show(prepared.attribute_links.head(8))
show(Markdown("**Flags per entity and type: the 8 largest**"))
show(prepared.flags.sort("count", descending=True).head(8))

## 3. A transparent offline embedder

`NetworkDetector.index` embeds the selected node types through any object with an
`embed_store_many` method, the slice of itk-core's `BaseEmbedder` the package relies on.
The adapter below lowercases a node's value, keeps letters and digits, and counts
character trigrams over the padded string. A sorted corpus vocabulary gives deterministic
aligned vectors, so the notebook needs no API key, model download or random seed.

This is a **lexical** baseline: it links spelling variants such as `Wind Power Inc` and
`WindPower Inc`, not synonyms or translations. Thresholds below are tuned for it and
would need recalibrating for a neural embedder. The embedder records how many times it is
called, which section 6 uses to show that re-running later stages never re-embeds.

In [ ]:
class OfflineTrigramEmbedder:
    def __init__(self) -> None:
        self.calls = 0

    @staticmethod
    def features(node: str) -> Counter[str]:
        value = node.split("==", 1)[-1]
        padded = f" {' '.join(re.findall(r'[a-z0-9]+', value.lower()))} "
        return Counter(padded[i : i + 3] for i in range(len(padded) - 2))

    async def embed_store_many(self, data, callbacks=None, cache_data=True):
        self.calls += 1
        features = [self.features(item["text"]) for item in data]
        vocabulary = {
            gram: i for i, gram in enumerate(sorted({g for c in features for g in c}))
        }
        vectors = np.zeros((len(data), len(vocabulary)), dtype=np.float32)
        for row, counts in zip(vectors, features, strict=True):
            for gram, count in counts.items():
                row[vocabulary[gram]] = count
        return [
            {**item, "vector": vector}
            for item, vector in zip(data, vectors, strict=True)
        ]


embedder = OfflineTrigramEmbedder()
detector = NetworkDetector(embedder)

## 4. Index entity names and infer similar ones

Indexing embeds every node of the chosen types (`ENTITY` here, the company names) and
stores each node's twenty nearest neighbours by cosine distance. `infer` then links pairs
whose distance is at most `similarity_threshold`; it only reads the stored neighbour
matrices. Inferred links join entities the way a shared attribute would, and are also used
when two attribute values are near-duplicates if attribute types are indexed too.

The workflow's tutorial suggests raising the threshold in small steps until the least
similar pairs shown at the top of the table stop looking like the same company. With the
trigram embedder, `0.15` keeps pairs that differ by a suffix, a space or a letter.

In [ ]:
indexed = await detector.index(prepared, ["ENTITY"])
print(
    f"Indexed {len(indexed.indexed_texts)} nodes; embedding matrix {indexed.embeddings.shape}"
)
print("Embedder calls so far:", embedder.calls)

result = detector.infer(indexed, similarity_threshold=0.15)
links = result.inferred_links_frame
show(
    Markdown(f"**Inferred links at threshold 0.15: first 15 of {links.height} pairs**")
)
show(links.head(15))
show(Markdown("**Entities with the most inferred links**"))
show(
    pl.concat([
        links.select(pl.col("text").alias("entity")),
        links.select(pl.col("similar").alias("entity")),
    ])
    .group_by("entity")
    .len()
    .sort(["len", "entity"], descending=[True, False])
    .head(8)
)

## 5. Identify networks

`NetworkConfig` holds the constraints the workflow exposes:

- **`max_attribute_degree=10`**: an attribute value shared by more than ten entities
  (a generic address, a shared switchboard number) says little about a close relationship
  and is ignored. The trimmed values are listed in `trimmed_attributes`.
- **`supporting_attribute_types=("city",)`**: a shared city alone does not connect two
  companies, but it is still shown when they are connected by other means.
- **`max_network_entities=20`**: connected components larger than this are split with
  hierarchical Leiden (graspologic) at `leiden_seed`, so each network stays small enough to
  reason about. The limit is a target: a tight clique may stay whole.

`identify` trims, projects the attribute graph onto entities (two entities are joined when
they are within three hops through usable attributes, equality edges or inferred links),
finds the networks largest first, and computes one record per entity with its network's
flag statistics: `network_flags`, the number of `flagged` entities, `flags/entity` and
`flagged/unflagged`. The entities frame is sorted by `flagged/unflagged`, the measure the
workflow ranks networks by; the grouped frame adds one column per group dimension.

In [ ]:
config = NetworkConfig(
    max_attribute_degree=10,
    max_network_entities=20,
    supporting_attribute_types=("city",),
)
identified = detector.identify(result, config)
print(identified.records_summary)
show(
    Markdown(
        f"**Attributes trimmed for high degree: {identified.trimmed_attributes.height}**"
    )
)
show(identified.trimmed_attributes.sort("Linked Entities", descending=True).head(10))

entities = identified.entities_frame
show(Markdown(f"**Entities frame: first 15 of {entities.height} rows**"))
show(entities.head(15))

network_columns = [
    "network_id",
    "network_entities",
    "network_flags",
    "flagged",
    "flags/entity",
    "flagged/unflagged",
]
networks = (
    entities.select(network_columns)
    .unique()
    .sort(["flagged/unflagged", "network_flags"], descending=True)
)
show(Markdown(f"**One row per network: first 10 of {networks.height}**"))
show(networks.head(10))

grouped = identified.grouped_frame
show(Markdown("**Grouped frame: entity records with sector and country**"))
show(grouped.head(10))
show(Markdown("**Flags by sector across all networks**"))
show(
    grouped.group_by("sector")
    .agg(pl.len().alias("entities"), pl.col("entity_flags").sum().alias("flags"))
    .sort("flags", descending=True)
    .head(8)
)

## 6. Change settings without re-embedding

Every stage returns a new `NetworkResult` and leaves its input untouched, keeping the
graph, the embeddings and the neighbour matrices. Re-running `infer` with another
threshold or `identify` with another configuration therefore costs seconds, not another
pass through the embedder. The table reports the observed effect of each setting: the
number of inferred links, networks found, entities in networks of more than one member
and the size of the largest network. The embedder call count stays at one throughout.

In [ ]:
calls_before = embedder.calls
sensitivity = []
for label, threshold, degree, max_entities in [
    ("strict links", 0.05, 10, 20),
    ("baseline", 0.15, 10, 20),
    ("loose links", 0.25, 10, 20),
    ("tighter attributes", 0.15, 5, 20),
    ("larger networks", 0.15, 10, 50),
]:
    variant = detector.identify(
        detector.infer(indexed, threshold),
        NetworkConfig(
            max_attribute_degree=degree,
            max_network_entities=max_entities,
            supporting_attribute_types=("city",),
        ),
    )
    assert variant.embeddings is indexed.embeddings
    sizes = variant.network_sizes
    sensitivity.append({
        "setting": label,
        "similarity_threshold": threshold,
        "max_attribute_degree": degree,
        "max_network_entities": max_entities,
        "inferred_links": variant.inferred_links_frame.height,
        "trimmed_attributes": variant.trimmed_attributes.height,
        "networks": len(variant.networks),
        "multi_entity_networks": len(sizes),
        "entities_in_them": sum(sizes),
        "largest_network": max(sizes, default=0),
    })
    if label == "baseline":
        assert variant.entities_frame.equals(identified.entities_frame)
assert embedder.calls == calls_before == 1
show(pl.DataFrame(sensitivity))
print("Embedder calls after five re-runs:", embedder.calls)

## 7. Explore one network

`NetworkDetector.explore` returns the attribute subgraph around one network: its entities,
their attributes, those attributes' attribute neighbours and one further hop, skipping
trimmed values. Nodes carry `type`, `flags` and, for entities, their `network`.
`simplify` drops attributes linked to a single entity and merges neighbouring attribute
nodes that share a type or a value (the inferred near-duplicates), which is the view the
AI report is written from. `subgraph_frames` turns either graph into node and edge frames;
colours and plot payloads are left to the caller.

The network chosen below is the one ranked first by `flagged/unflagged` among networks of
at least three entities.

In [ ]:
candidates = networks.filter(pl.col("network_entities") >= 3)
network_id = int(candidates["network_id"][0])
network = identified.network(network_id)
show(Markdown(f"**Network {network_id}: {network.size} entities**"))
show(
    entities.filter(pl.col("network_id") == network_id).sort(
        "entity_flags", descending=True
    )
)

full = NetworkDetector.explore(identified, network_id)
nodes, edges = subgraph_frames(full)
print(f"Full subgraph: {nodes.height} nodes, {edges.height} edges")
show(Markdown("**Full subgraph nodes by type**"))
show(nodes.group_by("type").len().sort("len", descending=True))

simplified = simplify(full)
simple_nodes, simple_edges = subgraph_frames(simplified)
show(
    Markdown(
        f"**Simplified subgraph: {simple_nodes.height} nodes, {simple_edges.height} edges**"
    )
)
show(simple_nodes.sort(["flags", "node"], descending=[True, False]))
show(simple_edges.sort(["source", "target"]))

## 8. Flag exposure of one entity

`NetworkDetector.exposure` takes the network and one entity (its unqualified id) and
follows the shortest paths from every flagged entity in the network subgraph to it. The
`summary` counts the entity's own `direct` flags, the `indirect` flags on the other
flagged entities, the number of distinct `paths` and of related `entities`. Paths that
share a route into the selected entity are grouped: the first step lists all their sources.
`markdown` renders the same report the workflow shows beside the graph; it is also what the
AI report step is given as context.

The entity chosen is the member of the network above with the most inferred links, or,
failing that, its most flagged member.

In [ ]:
members = entities.filter(pl.col("network_id") == network_id)
linked = members.join(
    links.select(pl.col("text").alias("entity_id"))
    .vstack(links.select(pl.col("similar").alias("entity_id")))
    .group_by("entity_id")
    .len(),
    on="entity_id",
    how="left",
).fill_null(0)
entity = linked.sort(
    ["len", "entity_flags", "entity_id"], descending=[True, True, False]
)["entity_id"][0]

exposure = NetworkDetector.exposure(identified, network_id, entity)
print(f"Selected entity: {entity}")
print(exposure.summary)
show(
    pl.DataFrame({
        "path": [i + 1 for i in range(len(exposure.paths))],
        "sources": [", ".join(p.steps[0]) for p in exposure.paths],
        "steps": [" -> ".join(step[0] for step in p.steps[1:]) for p in exposure.paths],
    })
)
show(Markdown(exposure.markdown))

## 9. The AI report, shown but skipped without an OpenAI configuration

`NetworkDetector.report_messages` builds the exact prompt the workflow sends: the
simplified network's node and edge CSVs, the exposure report, the network-wide maximum and
mean flag counts, and the safety instructions. `NetworkDetector.report` passes it to any
async chat function; `NetworkDetector.from_openai` wires in itk-core's `OpenAIClient`.

The cell inspects the prompt and only calls the model when `OPENAI_API_KEY` (or an Azure
endpoint under `AZURE_OPENAI_ENDPOINT`) is set. The call costs money and the text differs
between runs, so the saved notebook keeps the step skipped.

In [ ]:
messages = NetworkDetector.report_messages(identified, network_id, entity)
for message in messages:
    print(f"{message['role']}: {len(message['content'])} characters")
print()
print(messages[-1]["content"][:600], "...")

if os.environ.get("OPENAI_API_KEY") or os.environ.get("AZURE_OPENAI_ENDPOINT"):
    from itk_core.openai_configuration import OpenAIConfiguration

    reporter = NetworkDetector.from_openai(OpenAIConfiguration())
    report = await reporter.report(identified, network_id, entity)
    show(Markdown(report))
else:
    print("\nSkipped the LLM call: no OpenAI configuration in the environment.")

## Interpretation and limitations

- Networks are proposals built from shared attribute values and similar spellings, not
  verified relationships. A shared phone number or owner is strong evidence; a shared
  city is deliberately only supporting. Review the full and simplified subgraphs before
  treating a network as one organisation.
- The trigram embedder links spelling variants only. It will miss renamed companies and
  may join unrelated names that share many trigrams. Install `itk-core[local]` and pass
  `LocalEmbedder()` for neural local embeddings, or use `NetworkDetector.from_openai` for
  hosted ones; both need the threshold recalibrated.
- `max_network_entities` is a target. Leiden leaves a component whole when it cannot split
  it, and the split depends on `leiden_seed`; the package sorts nodes and edges before
  calling it so the same inputs give the same networks.
- Flag statistics count entities reached through inferred links into a network's totals
  once each, matching the workflow. Exposure paths run inside the explore subgraph, which
  does not keep equality edges between attributes, so an entity joined to the network only
  through such an edge contributes no path.
- The package returns frames and graphs, not figures. The Streamlit workflow draws the
  same `subgraph_frames` with its own colours.